# Tutorial: Live Ollama Five-Agent Demo

Audience:
- developers recording a real local-model Knoema demo without cloud keys.

Prerequisites:
- Ollama is installed and `ollama serve` is running.
- `llama3.1:8b` is available locally.
- The repo is installed in editable mode or the notebook can run `pip install -e .`.

Learning goals:
- verify the local Ollama model list before recording;
- run one deterministic seeded five-agent scenario;
- confirm the live path finishes under the 30-second cap;
- note that Ollama exposes Llama 3.3 as 70B, so the 8B live path uses `llama3.1:8b`.
- preview the JSONL artifact used for playback and QA.


## Outline

1. Install or import the repo helpers.
2. Check that `llama3.1:8b` is available in Ollama.
3. Run the seeded five-agent live demo with a 30-second cap.
4. Inspect the summary and JSONL preview before recording.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("knoema") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

from knoema.demo.ollama_live import (
    DEFAULT_OLLAMA_MODEL,
    DEFAULT_OLLAMA_SEED,
    available_ollama_models,
    run_live_ollama_demo,
    warm_live_ollama_demo_model,
)

ollama_model = os.getenv("KNOEMA_OLLAMA_LIVE_MODEL", DEFAULT_OLLAMA_MODEL)
seed = DEFAULT_OLLAMA_SEED
deadline_seconds = 30.0
installed_models = available_ollama_models()
warmup_seconds = warm_live_ollama_demo_model(model=ollama_model) if ollama_model in installed_models else None

{
    "ollama_model": ollama_model,
    "seed": seed,
    "deadline_seconds": deadline_seconds,
    "installed_models": installed_models,
    "warmup_seconds": warmup_seconds,
}


## Step 1 - Run the live five-agent pass

This cell executes one seeded tick. The helper uses `llama3.1:8b`, a fixed seed `20260419`, low-variance generation settings, and a strict 30.0-second threshold for the whole pass.


In [ ]:
if ollama_model not in installed_models:
    raise RuntimeError(f"Missing {ollama_model!r}. Run `ollama pull {ollama_model}` first.")

result = run_live_ollama_demo(
    model=ollama_model,
    seed=seed,
    deadline_seconds=deadline_seconds,
)

summary = {
    "model": result.model,
    "seed": result.seed,
    "tick_count": result.tick_count,
    "action_count": result.action_count,
    "relationship_edges": result.relationship_edges,
    "elapsed_seconds": result.elapsed_seconds,
    "completed_within_target": result.completed_within_target,
    "action_mix": result.action_mix,
}
summary


## Step 2 - Preview the JSONL artifact

Keep this short on screen. One or two rows are enough for the recording cue sheet.


In [ ]:
result.jsonl.splitlines()[:2]
